# Places

Analysis of places in the individual biographies.

This notebook collects all the places mentioned in the individual biographies and uses wikidata to extract associated information about them, namely current names, coordinates, and adminstrative information (present day).

Relevant to this notebook is the list of attribute types that receive place names as values.

The list is the following:

- 'nascimento'
- 'jesuita-entrada'
- 'partida'
- 'chegada'
- 'estadia'
- 'estadia-x'
- 'jesuita-votos-local'
- 'jesuita-ordenacao-padre'
- 'morte'




In [1]:
from timelink.notebooks import TimelinkNotebook

tlnb = TimelinkNotebook()
tlnb.print_info()

Timelink version: 1.1.33
Project name: dehergne
Project home: /Users/jrc/mhk-home/sources/dehergne
Database type: sqlite
Database name: dehergne
Kleio image: timelinkserver/kleio-server
Kleio server token: z0e25...
Kleio server URL: http://127.0.0.1:8090
Kleio server home: /Users/jrc/mhk-home/sources/dehergne
Kleio server container: adoring_curie
Kleio version requested: latest
Kleio server version: 12.9.591 (2026-02-17 05:25:12)
SQLite directory: /Users/jrc/mhk-home/sources/dehergne/database/sqlite
Database version: 6ccf1ef385a6
Call print_info(show_token=True) to show the Kleio Server token
Call print_info(show_password=True) to show the Postgres password
TimelinkNotebook(project_name=dehergne, project_home=/Users/jrc/mhk-home/sources/dehergne, db_type=sqlite, db_name=dehergne, kleio_image=timelinkserver/kleio-server, kleio_version=latest, postgres_image=postgres, postgres_version=latest)


## Define attributes types that have locations as values.

In [2]:
attributes_with_place_names = ['nascimento','jesuita-entrada','partida','chegada','estadia','estadia-x',
                'jesuita-votos-local','jesuita-ordenacao-padre','morte']

attributes_with_wikidata = [f"{attr}@wikidata" for attr in attributes_with_place_names]
attributes_with_wikidata

['nascimento@wikidata',
 'jesuita-entrada@wikidata',
 'partida@wikidata',
 'chegada@wikidata',
 'estadia@wikidata',
 'estadia-x@wikidata',
 'jesuita-votos-local@wikidata',
 'jesuita-ordenacao-padre@wikidata',
 'morte@wikidata']

Collect all the individuals with attributes related to locations. Make note of the dates of the attributes and infer dates when missing, based on last known date for a previous location attribute. Extract wikidata ids for the locations if available.



In [3]:
import pandas as pd
from timelink.pandas import entities_with_attribute
from timelink.kleio.utilities import format_timelink_date
# show 500 rows
pd.set_option('display.max_rows', 150)

places_of_stay = entities_with_attribute(
    entity_type='person',
    show_elements=['name','groupname','the_line','the_order','extra_info'],
    the_type=['jesuita-entrada','partida','chegada','estadia','estadia-x',
                'jesuita-votos-local','jesuita-ordenacao-padre','morte'],
    column_name='place',
    more_attributes=['jesuita-entrada','nascimento'],
    # filter_by=ids,
    db=tlnb.db,
)

# fillna with "?"
places_of_stay.place = places_of_stay.place.fillna('?')

# remove rows where place == '?'
places_of_stay = places_of_stay[places_of_stay.place != '?']

# this sequence replaces missing dates with the value of the previous date
# followed by ">". This allows to use the information that although the date
# is unkown it has happened after a certain date
# we need the id in a column to group by it (it is currently in the index)
# TODO: maybe an option in entities_with_attribute and compute intervals using
#       post quem and ante quem dates. See https://github.com/time-link/timelink-kleio/issues/1

places_of_stay['id_col'] = places_of_stay.index

# We try to infer the date of stay from the previous date known
# create a copy of the date column to replace the '0' values with NaN
places_of_stay['place.date_inferred'] = places_of_stay['place.date'].replace('0', pd.NA)

# order by id and line to have the missing date values filled with the previous date
places_of_stay = places_of_stay.sort_values(by=['id_col', 'place.line'])
# use ffill to fill the missing values with the previous value
places_of_stay['place.date_inferred'] = places_of_stay.groupby(['id_col'])['place.date_inferred'].ffill()
# create a column to flag the inferred dates
places_of_stay['place.date_is_inferred'] = places_of_stay['place.date_inferred'] != places_of_stay['place.date']
# reset index
places_of_stay.reset_index(inplace=True)
# if data_is_inferred set date_inferred to date_inferrred+">"
places_of_stay.loc[places_of_stay['place.date_is_inferred'], 'place.date_inferred'] = places_of_stay['place.date_inferred'] + '>'
# set missing value to ''
places_of_stay['place.date_inferred'] = places_of_stay['place.date_inferred'].fillna('')
# restore index
places_of_stay['place.date_inferred.formatted'] = places_of_stay['place.date_inferred'].apply(format_timelink_date)  
places_of_stay.index = places_of_stay['id_col']

places_of_stay.info()


<class 'pandas.core.frame.DataFrame'>
Index: 6297 entries, deh-abraham-le-royer to simao-rodrigues-ref1
Data columns (total 39 columns):
 #   Column                         Non-Null Count  Dtype 
---  ------                         --------------  ----- 
 0   id                             6297 non-null   object
 1   name                           6297 non-null   object
 2   groupname                      6297 non-null   object
 3   the_line                       6297 non-null   int64 
 4   the_order                      6297 non-null   int64 
 5   extra_info                     6297 non-null   object
 6   place.attr_id                  6297 non-null   object
 7   place.type                     6297 non-null   object
 8   place                          6297 non-null   object
 9   place.date                     6297 non-null   object
 10  place.line                     6297 non-null   int64 
 11  place.level                    6297 non-null   int64 
 12  place.groupname                6

## Function to extract wikidata ID

In [3]:
import pandas as pd
import re

def extract_wikidata_id(extra_info_dict):
    """
    Extract wikidata ID from place.extra_info JSON string.
    Returns the wikidata ID (e.g., 'Q6625') or NaN if not present.
    """
    if pd.isna(extra_info_dict):
        return pd.NA
    
    try:
        comment = extra_info_dict.get('the_value', {}).get('comment', '')
        if comment and 'wikidata:' in comment:
            # Extract wikidata ID using regex pattern (Q followed by digits)
            match = re.search(r'wikidata:(Q\d+)', comment)
            if match:
                return match.group(1)
        return pd.NA
    except (ValueError, SyntaxError, TypeError):
        
        return pd.NA


In [5]:
# Apply the function to create the new column
places_of_stay['place.wikidata_id'] = places_of_stay['place.extra_info'].apply(extract_wikidata_id)

## Show locations and wilidata Ids

In [6]:

# show results
show_only=10
cols=['place','place.wikidata_id','place.date_inferred.formatted','place.date_inferred','groupname','name','place.type', 'nascimento', 'nascimento.date', 'place.type','jesuita-entrada', 'jesuita-entrada.date', 'place.date_is_inferred','place.line','place.extra_info']
places_of_stay[places_of_stay.groupname=='n'][cols].sort_values(by=['place','place.date_inferred','name',]).head(show_only)



,place,place.wikidata_id,place.date_inferred.formatted,place.date_inferred,groupname,name,place.type,nascimento,nascimento.date,place.type,jesuita-entrada,jesuita-entrada.date,place.date_is_inferred,place.line,place.extra_info
id_col,,,,,,,,,,,,,,,
deh-charles-francois-xavier-de-brevedent,Abissínia,Q207521,1698-06-10,16980610,n,Charles-François Xavier de Brévedent,partida,"Genneville, Calvados",16590525,partida,Paris,16740909,False,1358,"{'the_value': {'kleio_element_name': 'valor', ..."
deh-antonio-sedeno,"Acapulco, México",Q81398,1581-03-29,15810329,n,Antonio Sedeño,estadia,"San Clemente, diocese de Cuenca",1535,estadia,Lorette,15590425,False,687,"{'id': {'kleio_element_name': 'id', 'kleio_ele..."
deh-antoine-rene-de-brasle,"Acapulco, México",Q81398,<1707,<1707,n,Antoine-René de Brasle,estadia,"Château-Thierry, diocese de Soissons",16680815,estadia,?,16851006,False,1297,"{'the_value': {'kleio_element_name': 'valor', ..."
deh-antonio-de-magalhaes,Afeganistão,Q889,1700,17000000,n,António de Magalhães,estadia,"Cortiços, diocese de Viseu",16770000,estadia,Coimbra,16921019,False,145,"{'the_type': {'kleio_element_name': 'tipo', 'k..."
deh-pierre-albier,Agen,Q6625,1654-07-12,16540712,n,Pierre Albier,jesuita-votos-local,Limousin,16200000,jesuita-votos-local,?,16440000,False,294,"{'the_value': {'kleio_element_name': 'valor', ..."
deh-philippe-avril,Agen,Q6625,1691,16910000,n,Philippe Avril,estadia,Angoulême (paróquia de Notre-Dame de Beaulieu),16540721,estadia,Bordeaux,16700916,False,1442,"{'the_value': {'kleio_element_name': 'valor', ..."
deh-bento-de-gois,Agra,Q42941,1602-10-29,16021029,n,Bento de Góis,estadia,"Vila Franca do Campo, ilha de S. Miguel, Açores",1563,estadia,?,1584,False,614,"{'the_value': {'kleio_element_name': 'valor', ..."
deh-bento-de-gois,Agra,Q42941,1602-10-29,16021029,n,Bento de Góis,estadia,"Vila Franca do Campo, ilha de S. Miguel, Açores",1563,estadia,?,1588,False,614,"{'the_value': {'kleio_element_name': 'valor', ..."
deh-antonio-de-andrade,Agra,Q42941,1624-03-30,16240330,n,António de Andrade,estadia,"Oleiros, diocese de Portalegre",15800000,estadia,Coimbra,15961216,False,1063,"{'the_value': {'kleio_element_name': 'valor', ..."


### Save to Excel

In [17]:
places_of_stay[cols].sort_values(by=['place','place.date_inferred']).to_excel(f"../inferences/jesuita_entrada_places_of_stay.xlsx", sheet_name='wikidata', index=True)
# another version without wikidata_id for unbiased LLM reasoning
places_of_stay[['place','place.date_inferred.formatted', 'name','place.type','place.original']].sort_values(by=['place','place.date_inferred.formatted']).to_excel(f"../inferences/locations_names_places_dates_only.xlsx", sheet_name='wikidata', index=True)

### Locations with no wikidata id

In [18]:
show_only=10
places_of_stay[(places_of_stay['place.wikidata_id'].isna())][cols].sort_values(by=['place','place.date_inferred']).head(show_only)

,place,place.wikidata_id,place.date_inferred.formatted,place.date_inferred,groupname,name,place.type,nascimento,nascimento.date,place.type,jesuita-entrada,jesuita-entrada.date,place.date_is_inferred,place.line,place.extra_info
id_col,,,,,,,,,,,,,,,
deh-miguel-vieira,Aquilon,<NA>,,,n,Miguel Vieira,estadia,"Pedroso, diocese do Porto",16810929,estadia,Lisboa,17020912,True,1301,"{'the_value': {'kleio_element_name': 'valor', ..."
deh-simao-da-silveira,Aquilon,<NA>,,,n,Simão da Silveira,estadia,"Borba, diocese de Évora",16941103,estadia,Évora,17090311,True,1406,"{'id': {'kleio_element_name': 'id', 'kleio_ele..."
deh-manuel-jose-ref2,Ascitano,<NA>,1765,17650000,referido,Emmanuel Josephus,morte,Marinha Grande,17020000,morte,Lisboa,17180000,False,231,"{'the_value': {'kleio_element_name': 'valor', ..."
deh-mathurin-lamathe,"Chang-tsin do Yüansi, Hou-pei",<NA>,1786-12-03,17861203,n,Mathurin Lamathe,morte,Aquitânia,17231111,morte,Bordeaux,17410724,False,332,"{'the_date': {'kleio_element_name': 'date', 'k..."
deh-mathurin-lamathe,"Chayüankow, Hukwang",<NA>,1759-04-01,17590401,n,Mathurin Lamathe,jesuita-votos-local,Aquitânia,17231111,jesuita-votos-local,Bordeaux,17410724,False,332,"{'the_date': {'kleio_element_name': 'date', 'k..."
deh-pierre-tchen,Choui tcheou San Kia (capela da Beata Virgem M...,<NA>,1745-09-08,17450908,n,Pierre T'chen,jesuita-votos-local,"Soochow, Kiang-nan",16881130,jesuita-votos-local,?,17340803,False,244,"{'the_value': {'kleio_element_name': 'valor', ..."
deh-andre-da-costa,"Cuamani, província de Goa",<NA>,1722-07-28,17220728,n,André da Costa,morte,Braga,1662,morte,?,1678,False,2108,"{'the_type': {'kleio_element_name': 'tipo', 'k..."
deh-magnani-ref1,Czerczerse,<NA>,1794-04-21,17940421,referido,Agostino Magnani,morte,Bolonha,000000,morte,?,17621012,False,334,"{'the_type': {'kleio_element_name': 'tipo', 'k..."
deh-diogo-vidal,"Hien, Tonquim",<NA>,1695-02-24,16950224,n,Diogo Vidal,jesuita-votos-local,"Cerradas, diocese de Lisboa",16600423,jesuita-votos-local,Lisboa,16780211,False,1173,"{'the_value': {'kleio_element_name': 'valor', ..."


#### Export to Excel all unlocated places

In [ ]:
places_of_stay[places_of_stay['wikidata_id']=='no wikidata'][cols].sort_values(by=['place','place.date_inferred']).to_excel(f"../inferences/locations_names_no_wikidata.xlsx", sheet_name='no_wikidata', index=True)

Export the list of wikidata ids for later processing by wikidata-linked-data notebook.

In [ ]:
# save to csv wikidata column, except when value is "No wikidata"
places_of_stay[places_of_stay['wikidata_id'] != 'no wikidata']['wikidata_id'].drop_duplicates().to_csv(f"../inferences/wikidata-references/locations_names_wikidata.csv", index=False)

#### Only specific place of entry, unkown places

In [ ]:
from timelink.pandas import entities_with_attribute

# Choose the place of entry
place_of_entry = 'Coimbra'

# get the ids of the places of entry
missionaries_from_place = entities_with_attribute(
    entity_type='person',
    show_elements=['name','groupname','the_line','the_order','extra_info'],
    the_type=['jesuita-entrada'],
    the_value=place_of_entry,
    db=tlnb.db,
)
print(f"Number of missionaries from {place_of_entry}: {len(missionaries_from_place)}")
place_of_entry_ids = missionaries_from_place.index
places_of_stay_specific = places_of_stay.loc[place_of_entry_ids]
places_of_stay_specific[places_of_stay_specific['wikidata_id'] == 'No wikidata'][cols].sort_values(by=['place','place.date_inferred'])

Number of missionaries from Coimbra: 62


,place,wikidata_id,place.date_inferred,place.date_inferred.formatted,groupname,name,place.type,place.comment,place.original,place.date_is_inferred,place.line,place.extra_info
id,,,,,,,,,,,,


In [ ]:
places_of_stay_specific[places_of_stay_specific['wikidata_id'] == 'No wikidata'][cols].sort_values(by=['place','place.date_inferred']).to_excel(f"../inferences/jesuita_entrada_{place_of_entry}_places_of_stay_no_wikidata.xlsx", sheet_name='no_wikidata', index=True)

### Frequency of places / Frequência dos lugares

In [ ]:
places_of_stay.info()

<class 'pandas.core.frame.DataFrame'>
Index: 7177 entries, deh-abraham-le-royer to simao-rodrigues-ref1
Data columns (total 23 columns):
 #   Column                  Non-Null Count  Dtype 
---  ------                  --------------  ----- 
 0   id                      7177 non-null   object
 1   name                    7177 non-null   object
 2   groupname               7177 non-null   object
 3   the_line                7177 non-null   int64 
 4   the_order               7177 non-null   int64 
 5   extra_info              7177 non-null   object
 6   place.attr_id           7177 non-null   object
 7   place.type              7177 non-null   object
 8   place                   7177 non-null   object
 9   place.date              7177 non-null   object
 10  place.line              7177 non-null   int64 
 11  place.level             7177 non-null   int64 
 12  place.obs               7177 non-null   object
 13  place.extra_info        7177 non-null   object
 14  place.comment           70

In [ ]:
col = 'place' # subtotal by this column

# Use pandas groupby and specify unique value count for id
df_totals = places_of_stay.groupby(col).agg({'id':'nunique',
                                                'place.date':'min',
                                                'place.date':'max'})
df_totals.info()
df_totals.sort_values('id',ascending= False).head(130)

<class 'pandas.core.frame.DataFrame'>
Index: 1569 entries, 's-Hertogenbosch, Holanda to Žilina, Eslováquia
Data columns (total 3 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   id                   1569 non-null   int64 
 1   place.date_inferred  1569 non-null   object
 2   place.date           1569 non-null   object
dtypes: int64(1), object(2)
memory usage: 49.0+ KB


,id,place.date_inferred,place.date
place,,,
Macau,495,,>1652
Pequim,255,,>17410115
Goa,209,,>1713
Cantão,200,,>16590123
China,186,,18050114
Lisboa,173,,17810300
Roma,110,,>16851112
Coimbra,89,,<16230324
Nanquim,68,,1780


In [ ]:
# export totals to excel
df_totals.sort_values('id',ascending= False).to_excel(f"../inferences/locations_totals.xlsx", sheet_name='totals', index=True)

### Who was at a specific place


List those who stayed at a specific place



In [ ]:
place="Coimbra"

In [ ]:
cols = ['place','name','place.type','place.date_inferred','place.obs']
show_only=100
places_of_stay[places_of_stay.place==place][cols].sort_values(by=['place.date_inferred']).head(show_only)


,place,name,place.type,place.date_inferred,place.obs
id_col,,,,,
deh-nicolas-trigault,Coimbra,Nicolas Trigault,estadia-x,,
deh-bartolomeo-tedeschi,Coimbra,Bartolomeo Tedeschi,estadia-x,,
deh-belchior-miguel-carneiro-leitao,Coimbra,Belchior Miguel Carneiro Leitão,nascimento,1519,
deh-pedro-martins,Coimbra,Pedro Martins,nascimento,15420000,
deh-pedro-de-alcacova,Coimbra,Pedro de Alcáçova,jesuita-entrada,15420000,saiu a primeira vez
deh-belchior-nunes-barreto,Coimbra,Belchior Nunes Barreto,jesuita-entrada,15430311,
deh-belchior-miguel-carneiro-leitao,Coimbra,Belchior Miguel Carneiro Leitão,jesuita-entrada,15430425,
deh-francisco-perez,Coimbra,Francisco Pérez,jesuita-entrada,15440125,
deh-tiburcio-de-quadros,Coimbra,Tibúrcio de Quadros,jesuita-entrada,15440418,


#### Export to markdown

In [ ]:
places_of_stay.itertuples(index=True)

In [ ]:
import os
from pandas import DataFrame
from timelink.pandas import group_attributes as person_attributes

# create if it does not exist directory ../inferences/{place_of_entry}/markdown"
directory = f"../inferences/{place_of_entry}/markdown"
if not os.path.exists(directory):
    os.makedirs(directory)


for id, name in places_of_stay[places_of_stay.place==place][['name']].sort_values('name').itertuples(index=True):
    with tlnb.db.session() as session:
        p = tlnb.db.get_person(id, session=session)
        session.add(p)
        pk:str = p.to_kleio()
        pk = f"""## {name}
^[In Edit mode, select title (including the ##), right click and "Extract selection"]

---
### {name} @ timelink
```
{pk}
```

---
Select title, right click on selected title, choose extract note
"""
        fname = f"{directory}/{id}.md"
        with open(fname, 'w', encoding='utf-8') as f:
            f.write(pk)

